In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# Text Pre-Processing

## 1. Lowercase

In [3]:
df["review"] = df["review"].str.lower()

In [4]:
df.iloc[0]["review"]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.<br /><br />the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.<br /><br />it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />i would say the main appeal of the show is due to the fa

## 2. Removing HTML Tags

In [5]:
import re
def removeHTMLTags (text):
    return re.sub(r"<.*?>", "", text)

df["review"] = df["review"].apply(removeHTMLTags)

In [6]:
df.iloc[0]["review"]

"one of the other reviewers has mentioned that after watching just 1 oz episode you'll be hooked. they are right, as this is exactly what happened with me.the first thing that struck me about oz was its brutality and unflinching scenes of violence, which set in right from the word go. trust me, this is not a show for the faint hearted or timid. this show pulls no punches with regards to drugs, sex or violence. its is hardcore, in the classic use of the word.it is called oz as that is the nickname given to the oswald maximum security state penitentary. it focuses mainly on emerald city, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. em city is home to many..aryans, muslims, gangstas, latinos, christians, italians, irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.i would say the main appeal of the show is due to the fact that it goes where other shows wo

## 3. Removing URLs

In [7]:
def removeURL(text):
    return re.sub(r"http\S+", "", text)

df["review"] = df["review"].apply(removeURL)

## 4. Removing Punctuations

In [8]:
def removePunctuations(text):
    return re.sub(r"[^a-zA-Z0-9\s]", "", text)

df["review"] = df["review"].apply(removePunctuations)

In [9]:
df.iloc[0]["review"]

'one of the other reviewers has mentioned that after watching just 1 oz episode youll be hooked they are right as this is exactly what happened with methe first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the wordit is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda em city is home to manyaryans muslims gangstas latinos christians italians irish and moreso scuffles death stares dodgy dealings and shady agreements are never far awayi would say the main appeal of the show is due to the fact that it goes where other shows wouldnt dare forget pretty pictur

## 5. Removing Stopwords & Executing stemming

We will use `nltk` (Natural Language Tool Kit) for this.

In [10]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [11]:
def stopword_stemming (text): # returns the tokens as list
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words("english"))
    ps = PorterStemmer()
    cleaned_tokens = [ps.stem(word) for word in tokens if word not in stop_words]
    return cleaned_tokens

In [12]:
df["cleaned_tokens"] = df["review"].apply(stopword_stemming)

# Mapping words to integers

In [13]:
all_words = [word for tokens in df["cleaned_tokens"] for word in tokens]

In [14]:
from collections import Counter
word_counts = Counter(all_words)

In [15]:
len(word_counts)

181471

In [16]:
vocab_size = 20000

In [17]:
vocab = sorted(word_counts, key=word_counts.get, reverse=True)[:vocab_size]

In [18]:
word_to_int = {word : i + 1 for i, word in enumerate(vocab)}

In [19]:
def encode_to_int (tokens):
    return [word_to_int[token] for token in tokens if token in word_to_int]

In [20]:
df["encoded review"] = df["cleaned_tokens"].apply(encode_to_int)

# Pre-padding
- Prepadding is done so that machine does not forget the review.

In [21]:
print(df["encoded review"].apply(len).mean())
print(df["encoded review"].apply(len).mode())
print(df["encoded review"].apply(len).max())
print(df["encoded review"].apply(len).min())

114.36252
0    62
Name: encoded review, dtype: int64
1357
2


In [22]:
seq_length = 500

import numpy as np
def pad_features (encoded_reviews, seq_length):
    features = np.zeros((len(encoded_reviews), seq_length), dtype=int)
    for i, row in enumerate(encoded_reviews):
        if seq_length < len(row):
            features[i, :] = np.array(row)[:seq_length]
        else:
            features[i, -len(row):] = np.array(row)
    return features

In [23]:
X = pad_features(df["encoded review"].values, seq_length)

# DataLoaders

In [24]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df["sentiment"])

In [25]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
import torch
from torch.utils.data import TensorDataset, DataLoader

train_data = TensorDataset(torch.from_numpy(X_train).long(), torch.from_numpy(y_train).float())
test_data = TensorDataset(torch.from_numpy(X_test).long(), torch.from_numpy(y_test).float())

In [27]:
train_loader = DataLoader(train_data, shuffle=True, batch_size=64)
test_loader = DataLoader(test_data, batch_size=64)

# Defining the architecture

In [28]:
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, num_layers):
        super().__init__()

        self.num_layers = num_layers
        self.hidden_size = hidden_size
        
        # Embedding -> Changing each word (number currently) to a embedding vector which has dim embedding_dim

        self.embedding = nn.Embedding(num_embeddings=vocab_size+1, embedding_dim=embedding_dim) 
        # vocab_size +1 because of padded '0'

        self.rnn = nn.RNN(input_size=embedding_dim, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        # By default, RNN expect that shape of input is (sequence length, batch size, features/embedding_dim)
        # But we want (batch size, sequence length, features/embedding_dim). So, we specify batch_first=True.

        self.fc = nn.Linear(hidden_size, 1)
        # Since there are only '2' output classes. So, we our final layer has only one neuron.
        # In case of more classes, our output layers = number of classes. Then we apply CrossEntropyLoss

    def forward(self, x):
        # shape of x = (batch_size, seq_length)
        batch_size = x.size(0)

        embedded = self.embedding(x)
        # shape of embedded = ()

        h0 = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(x.device)

        out, _ = self.rnn(embedded, h0)
        # shape of out -> (batches, seq length, embedding dim)

        final_out = out[:, -1, :]

        ans = self.fc(final_out)

        return torch.sigmoid(ans).squeeze()

# Training

In [29]:
embedding_dim = 100
hidden_size = 128
num_layers = 2

model = RNN(vocab_size, embedding_dim, hidden_size, num_layers)
device = torch.device("mps")
model.to(device)

RNN(
  (embedding): Embedding(20001, 100)
  (rnn): RNN(100, 128, num_layers=2, batch_first=True)
  (fc): Linear(in_features=128, out_features=1, bias=True)
)

In [30]:
import torch.optim as optim

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

In [31]:
epochs = 10

best_eval_loss = float("inf")

for epoch in range (epochs):
    model.train()
    epoch_train_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        model.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()

    epoch_train_loss /= len(train_loader)

    model.eval()
    epoch_test_loss = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            outputs = model(xb)
            loss = criterion(outputs, yb)
            epoch_test_loss += loss.item()

    epoch_test_loss /= len(test_loader)
    if (epoch_test_loss < best_eval_loss):
        best_eval_loss = epoch_test_loss
        torch.save(model.state_dict(), "best_model.pt")
    print(f"{epoch + 1}/{epochs} : Train Loss: {epoch_train_loss}, Test Loss : {epoch_test_loss}")

1/10 : Train Loss: 0.6223870012283326, Test Loss : 0.6922574760807547
2/10 : Train Loss: 0.6527565305709839, Test Loss : 0.6416763105210225
3/10 : Train Loss: 0.5556965962886811, Test Loss : 0.5543373039193974
4/10 : Train Loss: 0.5815302670478821, Test Loss : 0.6537231941511676
5/10 : Train Loss: 0.5129158559799194, Test Loss : 0.48970991221203164
6/10 : Train Loss: 0.5489330462932587, Test Loss : 0.6058207554802014
7/10 : Train Loss: 0.4576617599964142, Test Loss : 0.4796019564768311
8/10 : Train Loss: 0.4338784512042999, Test Loss : 0.42983100824295334
9/10 : Train Loss: 0.3837904580593109, Test Loss : 0.6020493055604825
10/10 : Train Loss: 0.4684749782562256, Test Loss : 0.4725305857552085


In [32]:
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
correct_vals = 0
tot_vals = 0


with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        outputs = model(Xb)
        predicted = (outputs > 0.5).float()
        
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

print(f"Accuracy = {(correct_vals/tot_vals)*100}%")

Accuracy = 80.89%
